# 00 - Extract frozen features (tiny AND base)

Fixes the duplicated-file bug: `brainomni_features_base.npy` was a byte-identical copy of
the tiny file (both 8192-dim = base). Here we extract **genuine** frozen features for BOTH
models, each from its OWN checkpoint and its OWN cached `proj_out`, with explicit naming.

Reproduces `model.encode` exactly (tokenize -> +neuro -> projection -> `blocks[:-1]` ->
`F.normalize`), starting from the cached post-projection `proj_out`, then `.mean(dim=2)` over
time windows, average the two 30s segments, flatten. No raw `.mat` reprocessing, so it is fast.

Expected: tiny -> (148, 4096), base -> (148, 8192), and they must be DISTINCT.
Cross-check: base should reproduce the old 0.831; tiny is a genuinely new number.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import sys, json
import numpy as np, torch
import torch.nn.functional as F
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
from brainomni.model import BrainOmni

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
IN  = '../data/preprocessed'                 # cached proj_out + labels (deterministic inputs)
OUT = '../results_withseed'                  # corrected frozen features live here
CKPT = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection'
os.makedirs(OUT, exist_ok=True)
labels = np.load(f'{IN}/labels.npy'); N = len(labels)
print('Device:', device, '| N =', N)

W0815 00:13:55.303000 1406 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Device: mps | N = 148


In [2]:
def load_frozen_model(ckpt_dir):
    cfg = json.load(open(f'{ckpt_dir}/model_cfg.json'))
    m = BrainOmni(**cfg)
    m.load_state_dict(torch.load(f'{ckpt_dir}/BrainOmni.pt', map_location='cpu', weights_only=True), strict=False)
    for p in m.parameters(): p.requires_grad = False
    return m.to(device).eval()

def extract_frozen(model, proj1, proj2):
    """Per subject: run each 30s segment's proj_out through blocks[:-1] (matches model.encode),
    L2-normalize, mean over time windows, average the two segments, flatten -> (16*D,)."""
    feats = []
    with torch.no_grad():
        for i in range(N):
            x1 = torch.tensor(proj1[i]).unsqueeze(0).to(device)
            x2 = torch.tensor(proj2[i]).unsqueeze(0).to(device)
            for b in list(model.blocks)[:-1]:      # all but the last block == model.encode
                x1 = b(x1); x2 = b(x2)
            x1 = F.normalize(x1, p=2.0, dim=-1, eps=1e-6).mean(dim=2)   # over time windows
            x2 = F.normalize(x2, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
            feats.append(((x1 + x2) / 2).flatten().cpu().numpy())
    return np.stack(feats)

for tag, sfx in [('tiny', ''), ('base', '_base')]:
    p1 = np.load(f'{IN}/proj_out1{sfx}.npy'); p2 = np.load(f'{IN}/proj_out2{sfx}.npy')
    model = load_frozen_model(f'{CKPT}/{tag}')
    X = extract_frozen(model, p1, p2)
    np.save(f'{OUT}/frozen_{tag}.npy', X)
    print(f'{tag}: proj {p1.shape} -> frozen {X.shape}  |  saved {OUT}/frozen_{tag}.npy', flush=True)

/opt/miniconda3/envs/brainomni/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


tiny: proj (148, 16, 160, 256) -> frozen (148, 4096)  |  saved ../results_withseed/frozen_tiny.npy
base: proj (148, 16, 160, 512) -> frozen (148, 8192)  |  saved ../results_withseed/frozen_base.npy


In [3]:
# Cross-check: distinct? correct dims? does base reproduce the old 0.831?
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
Xt = np.load(f'{OUT}/frozen_tiny.npy'); Xb = np.load(f'{OUT}/frozen_base.npy')
print('tiny', Xt.shape, '| base', Xb.shape)
print('DISTINCT arrays?', not np.array_equal(Xt, Xb), '(must be True now)')
cv = StratifiedKFold(5, shuffle=True, random_state=123)
for name, X in [('frozen tiny', Xt), ('frozen base', Xb)]:
    acc = cross_val_score(LogisticRegression(max_iter=2000), X, labels, cv=cv, scoring='accuracy').mean()
    print(f'{name}: 5-fold CV LR acc {acc:.3f}')
print('\nOld buggy file gave 0.831 for BOTH. Base should land ~0.831; tiny is a NEW real number.')

tiny (148, 4096) | base (148, 8192)
DISTINCT arrays? True (must be True now)
frozen tiny: 5-fold CV LR acc 0.831
frozen base: 5-fold CV LR acc 0.831

Old buggy file gave 0.831 for BOTH. Base should land ~0.831; tiny is a NEW real number.
